In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_pickle("../../data/processed/19_cleaned_master_data.pkl")

# avg_speed Outlier Kontrolle

In [4]:
df["avg_speed"].describe()

count    196048.000000
mean         40.414783
std           3.770265
min          27.570000
25%          37.870000
50%          40.636000
75%          43.080000
max          51.230000
Name: avg_speed, dtype: float64

In [7]:
df9 = pd.read_pickle("../../data/processed/09_cleaned_master_data.pkl")
df9["avg_speed"].describe()

count    225539.000000
mean         41.197484
std           5.050417
min          18.666000
25%          38.066000
50%          41.000000
75%          43.813000
max          58.830000
Name: avg_speed, dtype: float64

In [8]:
def iqr_outliers(df: pd.DataFrame, cols: list[str], k: float = 1.5) -> pd.DataFrame:
    rows = []
    for col in cols:
        s = df[col].dropna()
        if s.empty:
            continue
        q1, q3 = np.percentile(s, [25, 75])
        iqr = q3 - q1
        lower = q1 - k * iqr
        upper = q3 + k * iqr
        outliers = s[(s < lower) | (s > upper)]
        rows.append(
            {
                "column": col,
                "n": int(s.shape[0]),
                "q1": q1,
                "q3": q3,
                "iqr": iqr,
                "lower_bound": lower,
                "upper_bound": upper,
                "min": s.min(),
                "max": s.max(),
                "n_outliers": int(outliers.shape[0]),
                "outlier_pct": round(100 * outliers.shape[0] / s.shape[0], 2),
            }
        )
    return (
        pd.DataFrame(rows)
        .set_index("column")
        .sort_values("outlier_pct", ascending=False)
    )


outlier_table = iqr_outliers(df, ["avg_speed"]).round(3)
outlier_table

,n,q1,q3,iqr,lower_bound,upper_bound,min,max,n_outliers,outlier_pct
column,,,,,,,,,,
avg_speed,196048,37.87,43.08,5.21,30.055,50.895,27.57,51.23,473,0.24
